# Stuttering Detection: Probabilistic Classification Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Naive Bayes vs. Normal Bayes (LDA)

---

## Step 1: Environment Setup
We use the project's standardized modular engine to load high-dimensional WavLM embeddings. Probabilistic models assume different distributions for the feature data.

In [ ]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import NaiveBayesModel, NormalBayesModel

# Paths to our distributed dataset
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Now using NATIVE Random Sampling logic for diversity
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Loading and Pipeline
We perform stratified splitting and utilize **Oversampling** to bring the rare disfluent samples in line with the fluent majority. This is critical for probabilistic models to learn accurate class priors.

In [ ]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading distributed .npy files
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Balancing classes for accurate probability estimation
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standard Scaling (Pivotal for Gaussian Bayes)
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

## Step 4: Model 1 - Naive Bayes
Naive Bayes assumes all features are independent. While this is rarely true in audio, it often provides a high-recall baseline.

In [ ]:
nb_model = NaiveBayesModel("Gaussian_Naive_Bayes")
nb_model.train(X_train_final, y_train_bal)
nb_model.evaluate(X_test_final, y_test)

## Step 5: Model 2 - Normal Bayes (LDA)
Normal Bayes (implemented via Linear Discriminant Analysis) accounts for the covariance between features, leading to more refined decision boundaries than Naive Bayes.

In [ ]:
lda_model = NormalBayesModel("Normal_Bayes_LDA")
lda_model.train(X_train_final, y_train_bal)
lda_model.evaluate(X_test_final, y_test)